<a href="https://colab.research.google.com/github/Azidian/Inteligencia_Artificial/blob/main/Componente_practico/Lecture_05/02_qlearning_frozenlake.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Trabajo en clase — Q-Learning con FrozenLake

En CheeseWorld construimos el algoritmo desde cero. Ahora utilizaremos el mismo procedimiento en un ambiente estándar de **Gymnasium**.

## Objetivo

Durante la clase debes relacionar cada parte del código con los conceptos:

- estado \(s\);
- acción \(a\);
- recompensa \(r\);
- Q-table;
- exploración y explotación;
- TD target;
- TD error;
- política greedy.

Este notebook tiene **espacios para discutir y escribir conclusiones durante la clase**.


## 1. Imports y funciones de Q-Learning


In [ ]:
import numpy as np
import gymnasium as gym
import random
import matplotlib.pyplot as plt
import time
from IPython.display import clear_output, display

from IPython.display import HTML
from matplotlib import animation
import matplotlib.pyplot as plt


def play_episode(env, Q=None, random_policy=False, max_steps=100, seed=None):
    """Ejecuta un episodio y devuelve sus frames y recompensa total."""
    state, _ = env.reset(seed=seed)
    frames = [env.render()]
    total_reward = 0.0

    for _ in range(max_steps):
        if random_policy:
            action = env.action_space.sample()
        else:
            q_values = Q[state]
            max_q = np.max(q_values)

            # Desempate aleatorio entre acciones con el mismo Q.
            best_actions = np.flatnonzero(q_values == max_q)
            action = int(np.random.choice(best_actions))

        next_state, reward, terminated, truncated, _ = env.step(action)

        frames.append(env.render())
        total_reward += reward
        state = next_state

        if terminated or truncated:
            break

    return frames, total_reward


def frames_to_video(frames, interval=700):
    """Convierte una lista de frames RGB en una animación reproducible en Jupyter."""
    fig = plt.figure(figsize=(4, 4))
    plt.axis("off")

    image = plt.imshow(frames[0])

    def update(frame):
        image.set_data(frame)
        return [image]

    anim = animation.FuncAnimation(
        fig,
        update,
        frames=frames,
        interval=interval,
        blit=True,
        repeat=True
    )

    plt.close(fig)
    return HTML(anim.to_jshtml())


def initialize_q_table(state_space, action_space):
    return np.zeros((state_space, action_space))


def greedy_policy(Qtable, state):
    # Si hay empate entre varias acciones con el mismo Q, desempata al azar.
    max_q = np.max(Qtable[state])
    best_actions = np.flatnonzero(Qtable[state] == max_q)
    return int(np.random.choice(best_actions))


def epsilon_greedy_policy(Qtable, state, epsilon, env):
    if random.random() < epsilon:
        return env.action_space.sample()

    return greedy_policy(Qtable, state)


def train_q_learning(
    env,
    Qtable,
    n_episodes=5000,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
    max_steps=100,
    start_episode=0,
):
    rewards = []

    for episode in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0

        global_episode = start_episode + episode
        epsilon = min_epsilon + (
            max_epsilon - min_epsilon
        ) * np.exp(-decay_rate * global_episode)

        for _ in range(max_steps):
            action = epsilon_greedy_policy(
                Qtable, state, epsilon, env
            )

            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            best_next_q = 0.0 if done else np.max(Qtable[next_state])

            td_target = reward + gamma * best_next_q
            td_error = td_target - Qtable[state, action]

            Qtable[state, action] += learning_rate * td_error

            state = next_state
            total_reward += reward

            if done:
                break

        rewards.append(total_reward)

    return Qtable, rewards


def evaluate_q_policy(env, Qtable, n_episodes=100, max_steps=100):
    episode_rewards = []

    for _ in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0

        for _ in range(max_steps):
            action = greedy_policy(Qtable, state)

            next_state, reward, terminated, truncated, _ = env.step(action)

            state = next_state
            total_reward += reward

            if terminated or truncated:
                break

        episode_rewards.append(total_reward)

    return np.mean(episode_rewards), np.std(episode_rewards)


def show_frame(env, title=''):
    frame = env.render()
    plt.figure(figsize=(4, 4))
    plt.imshow(frame)
    plt.axis('off')
    plt.title(title)
    display(plt.gcf())
    plt.close()

### 💬 Antes de ejecutar

En CheeseWorld teníamos explícitamente una clase `Environment` y una clase `QLearningAgent`.

**Pregunta:** en este notebook, ¿qué papel cumple Gymnasium y dónde quedó representado el agente?

**Respuesta:** Gymnasium cumple el rol del ambiente, es el que define las reglas del juego, la cuadrícula, la física, si hay piso resbaloso o no, y es el que recibe las acciones del agente para devolverle el siguiente estado ($s'$) y la recompensa ($r$).

Mientras que el agente no está encapsulado en una clase específica como en CheeseWorld, sino que está representado por la Q-table que es la que guarda su "memoria" y las funciones de política (epsilon_greedy_policy y greedy_policy) que toman las decisiones basándose en dicha tabla.


## 2. Crear FrozenLake


In [ ]:
env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=False,
    render_mode="rgb_array"
)

state, info = env.reset()

print("Estado inicial:", state)
print("Número de estados:", env.observation_space.n)
print("Número de acciones:", env.action_space.n)

En FrozenLake las acciones son:

| Acción | Código |
|---|---:|
| Left | 0 |
| Down | 1 |
| Right | 2 |
| Up | 3 |

Primero trabajaremos con `is_slippery=False`, es decir, con transiciones determinísticas.


### 💬 Actividad 1 — La Q-table

Antes de crearla:

1. ¿Cuántas filas debe tener la Q-table?

Debe tener 16 filas, correspondientes a los 16 estados posibles del mapa de 4x4 (state_space = 16).   

2. ¿Cuántas columnas?

Debe tener 4 columnas, que corresponden a las 4 acciones posibles (Left, Down, Right, Up) (action_space = 4).   

3. ¿Qué representa una celda $Q[s,a]$?

Representa el valor esperado o sea, la estimación de recompensa futura acumulada que el agente cree que obtendrá si estando en el estado $s$, decide ejecutar la acción $a$, y de ahí en adelante se sigue comportando de manera óptima.


In [ ]:
state_space = env.observation_space.n
action_space = env.action_space.n

Q = initialize_q_table(state_space, action_space)

print("Q-table shape:", Q.shape)
Q

### 💬 Actividad 2 — Inicio del aprendizaje

Todos los valores son cero.

$$
Q(s,a)=0
$$

- ¿Esto significa que todas las acciones son malas, o que el agente todavía no sabe nada?

Significa que el agente no sabe nada. Al no haber explorado el mapa, no tiene idea de qué acciones lo conducen al éxito o al fracaso, por lo que su expectativa para todas es cero, o sea, 0.

- ¿Qué ocurre si varias acciones tienen exactamente el mismo valor máximo?

El código utiliza un "desempate aleatorio". Como se observa en la función greedy_policy, si hay múltiples acciones empatadas (por ejemplo, al inicio todas valen 0), se elige una de ellas completamente al azar (np.random.choice(best_actions)).

## 3. Ejecutar una transición


In [ ]:
state, _ = env.reset()

epsilon = 1.0
action = epsilon_greedy_policy(Q, state, epsilon, env)

next_state, reward, terminated, truncated, _ = env.step(action)

print("state      =", state)
print("action     =", action)
print("reward     =", reward)
print("next_state =", next_state)
print("done       =", terminated or truncated)

### 💬 Actividad 3 — Identificar la experiencia

Escribe la experiencia anterior como:

$$
(s,a,r,s')
$$

**Experiencia:**

$$(0, 3, 0, 0)$$

¿De cuál de esos cuatro elementos **no disponíamos directamente** en Value Iteration cuando hablábamos de experiencia real?


En Value Iteration no interactuábamos con el mundo para obtener un resultado empírico, por lo que no teniamos un resultado del azar, es decir, durante Value Iteration no se contaba con el estado siguiente de manera específica ($s'$) al que se llegaba luego de ejecutar una acción, ni con la recompensa real observada ($r$) que el mundo devolvía. Todo se calculaba teóricamente usando el modelo de transición con sus probabilidades $P(s'\vert{}s,a)$ en vez de interactuar empíricamente con el entorno.   

## 4. Del azar a una política aprendida

Vamos a observar **el mismo agente en tres momentos**. Primero no sabe nada y actúa al azar; luego veremos su política después de pocas experiencias; finalmente veremos la política después del entrenamiento completo.


In [ ]:
# Guardaremos tres momentos del aprendizaje

# Momento 1: sin entrenamiento
Q_initial = initialize_q_table(
    env.observation_space.n,
    env.action_space.n
)

# Momento 2: poco entrenamiento
EARLY_EPISODES = 50
Q_early = Q_initial.copy()
Q_early, rewards_early = train_q_learning(
    env,
    Q_early,
    n_episodes=EARLY_EPISODES,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
)

# Momento 3: continuar hasta 10 000 episodios
TOTAL_EPISODES = 10000
Q_trained = Q_early.copy()
Q_trained, rewards_final = train_q_learning(
    env,
    Q_trained,
    n_episodes=TOTAL_EPISODES - EARLY_EPISODES,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
    start_episode=EARLY_EPISODES,
)

Q = Q_trained
rewards = rewards_early + rewards_final

print('Snapshots guardados:')
print('Q_initial : 0 episodios')
print(f'Q_early   : {EARLY_EPISODES} episodios')
print(f'Q_trained : {TOTAL_EPISODES} episodios')


### Momento 1 — Sin entrenamiento: random walk
Todavía no usamos la Q-table para decidir. Cada acción se selecciona aleatoriamente. Observa cómo interactúa el agente con el mundo.

In [ ]:
frames_random, reward_random = play_episode(
    env,
    Q_initial,
    random_policy=True,
    seed=7
)

print(f"Recompensa total: {reward_random}")
frames_to_video(frames_random, interval=700)


### Momento 2 — Después de pocas iteraciones

Ahora el agente usa de forma **greedy** lo que ha aprendido en `Q_early`. Todavía conoce poco del ambiente, así que su comportamiento puede ser incompleto o equivocarse.


In [ ]:
frames_early, reward_early = play_episode(
    env,
    Q_early,
    random_policy=False,
    seed=7
)

print(f"Recompensa total: {reward_early}")
frames_to_video(frames_early, interval=700)


### Momento 3 — Agente entrenado

Finalmente usamos `Q_trained`. Ya no exploramos: en cada estado el agente selecciona una de las acciones con mayor valor $Q(s,a)$.


In [ ]:
frames_trained, reward_trained = play_episode(
    env,
    Q_trained,
    random_policy=False,
    seed=7
)

print(f"Recompensa total: {reward_trained}")
frames_to_video(frames_trained, interval=700)


### 💬 Actividad — ¿Qué cambió?

Compara las tres ejecuciones. El ambiente, los estados y las acciones son los mismos. **¿Qué cambió internamente en el agente para que su comportamiento mejore?**

Observa `Q_initial`, `Q_early` y `Q_trained` y relaciona sus valores con las acciones que viste ejecutar.


Lo único que cambió fueron los valores numéricos dentro de la Q-table (Q_trained vs Q_initial). Al interactuar repetidamente y aplicar la ecuación de Bellman, el agente fue propagando las recompensas de la meta hacia atrás. Esto creó o generó la matriz para que, al elegir la acción con el valor más alto en cada fila, el agente trace un camino seguro evitando los agujeros.


In [ ]:
print(Q_trained[14])

### 💬 Actividad 4 — Leer una fila de Q

Selecciona un estado $s$ y observa:

$$
Q(s,0), Q(s,1), Q(s,2), Q(s,3)
$$

**Estado seleccionado:** _14_

**Valores Q:**

- Left: 0.9025
- Down: 0.95
- Right: 1.0
- Up: 0.9025

¿Cuál acción seleccionaría:
$$
\arg\max_a Q(s,a)
$$

?


Seleccionaría la acción 2 (Right), ya que es la que posee el valor más alto (1.0).

**Interpretación:**

> El estado 14 es la casilla que está inmediatamente a la izquierda de la meta en el mapa de 4x4. El agente ha aprendido perfectamente que moverse hacia la derecha desde esta posición lo lleva directo a la victoria, por lo que le asigna la máxima utilidad esperada (1.0). Las demás acciones tienen valores positivos altos porque de todas formas el agente ya sabe cómo ganar desde allí, pero son menores a 1.0 porque implicarían dar un paso innecesario alejándose temporalmente del objetivo.


## 5. Evaluar la política aprendida


In [ ]:
mean_reward, std_reward = evaluate_q_policy(
    env,
    Q_trained,
    n_episodes=100
)

print(f"Mean reward: {mean_reward:.3f}")
print(f"Std reward : {std_reward:.3f}")


### 💬 Actividad 5 — Exploration vs. exploitation

Durante entrenamiento usamos $\epsilon$-greedy.

Durante evaluación usamos:

$$
a=\arg\max_aQ(s,a)
$$

¿Por qué **no exploramos** durante la evaluación?

**Conclusión:**

> Porque durante la evaluación queremos medir qué tan inteligente se volvió el agente, es decir, qué tan buena es la política que aprendió. Si lo forzamos a la exploración, el agente se tiraría intencionalmente a los hoyos por el simple hecho de explorar, lo que arruinaría la medición de su rendimiento real. Evaluamos su "modo serio" (Explotación)


## 6. Experimento: FrozenLake estocástico


In [ ]:
slippery_env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=True,
    render_mode="rgb_array"
)

Q_slippery = initialize_q_table(
    slippery_env.observation_space.n,
    slippery_env.action_space.n
)

Q_slippery, rewards_slippery = train_q_learning(
    slippery_env,
    Q_slippery,
    n_episodes=20000,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.0005,
    max_steps=100
)

mean_reward, std_reward = evaluate_q_policy(
    slippery_env,
    Q_slippery,
    n_episodes=500
)

print(f"Mean reward: {mean_reward:.3f}")
print(f"Std reward : {std_reward:.3f}")

### 💬 Actividad 6 — Determinístico vs. estocástico

Compara:

- `is_slippery=False`
- `is_slippery=True`

¿Qué cambia en el **ambiente**?

El suelo ahora resbala (is_slippery=True). Esto significa que si el agente decide moverse hacia abajo, hay una alta probabilidad de que resbale y termine moviéndose hacia la izquierda o a la derecha de manera involuntaria.

¿Qué cambia en la **ecuación de Q-Learning**?


Nada cambia, porque Q-Learning se basa en la experiencia directa $(s,a,r,s')$, las probabilidades del piso resbaloso se irán promediando naturalmente a través del learning_rate ($\alpha$) a lo largo de miles de episodios, ajustando los valores Q sin que tengamos que modificar la ecuación


# Cierre de clase

Completa antes de terminar:

**1. ¿Qué almacena $Q(s,a)$?**

> La utilidad (recompensa futura esperada y con descuento) de tomar la acción $a$ desde el estado $s$ y seguir actuando óptimamente a partir de ahí.

**2. ¿De dónde sale $\max_{a'}Q(s',a')$?**

> Sale de observar la propia Q-table (esto se llama Bootstrapping). El agente mira el estado siguiente $s'$ al que acaba de llegar, escanea su tabla para ver cuál es la mejor acción $a'$ disponible allí, y toma ese valor como su expectativa del futuro.

**3. ¿Por qué necesitamos $\epsilon$-greedy?**

> Para equilibrar la exploración y la explotación. Si siempre eligiéramos la mejor acción conocida (explotación), el agente se quedaría con la primera ruta (así sea una ruta mediocre) que encontrara, es muy conservador y nunca descubriría caminos más eficientes o recompensas mejores.

**4. ¿Por qué Q-Learning es model-free?**

> Porque no necesita conocer ni las probabilidades de transición $P(s'\vert{}s,a)$ ni el mapa de recompensas $R(s)$ del entorno. Aprende a ciegas, basándose únicamente en experimentar repetidamente la tupla $(s,a,r,s')$.
